In [ ]:
# Bank Data Exploration

## 1. Raw data inspection
## 2. Data quality issues found
## 3. Cleaning decisions made
## 4. Category mapping approach
## 5. Final clean dataset verification

## This notebook explores the data from Revolut DKK transaction statements.
### The goal is to clean and combine data from different years and create a parquet file containing everything, for future analysis.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [ ]:
PROJECT_PATH = Path.cwd().parent
ARCHIVE_PATH = PROJECT_PATH / "archive"
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"

print(PROJECT_PATH)
print(ARCHIVE_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

#### --- DATA INSPECTION

In [ ]:
def inspect_dataset(bank: str, filename: str) -> list[Path]:
    """
    Inspect one raw CSV dataset for a given file name and return it as a data frame.
    
    Displays the first two and last rows, column types, and NaN counts for each CSV file found in the bank's raw data directory.

    Args:
        filename: the name of the file that contains the dataset.

    Returns:
        A data frame.

    Raises:
    """
    df = pd.read_csv(DATA_PATH / "raw" / bank /filename)
    
    display(df.head(2))
    display(df.tail(2))
    print("Types: ")
    display(df.dtypes)
    print("\n")
    print("Nans: ")
    display(df.isna().sum().to_dict())
    print("\n")

    return df
    

DF = inspect_dataset("revolut", "revolut_dkk_2024-2026.csv")

In [ ]:
DF[DF.isna().any(axis=1)]

In [ ]:
for c in ["Type", "Product", "Currency", "State"]:
    print(DF[c].unique(), end="\n\n\n")

In [ ]:
def clean_data_revolut(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["State"] != "REVERTED"]
    df = df.drop(columns=["Type", "Product", "Completed Date", "State"])
    df = df.rename(columns={
        "Started Date": "date",
        "Description": "description",
        "Amount": "amount",
        "Fee": "fee",
        "Currency": "currency",
        "Balance": "balance"
    })
    df["date"] = pd.to_datetime(df["date"])
    print("Data frame is cleaned.")
    return df

DF = clean_data_revolut(DF)

In [ ]:
def dkk_to_euro_column(df: pd.DataFrame) -> pd.DataFrame:
    RATE = 7.46
    df["amount_euro"] = (df["amount"] / RATE).round(2)
    return df

DF = dkk_to_euro_column(DF)

In [ ]:
def extract_descriptions(df: pd.DataFrame) -> None:
    
    print(f"{len(df['description'].unique())} unique descriptions detected.")
    
    descriptions = dict()
    for d in df["description"].unique():
        descriptions[d] = "blank"

    out_dir = ARCHIVE_PATH / "blank_mappings" / "revolut"
    out_dir.mkdir(parents=True, exist_ok=True)

    with open(out_dir / "revolut_dkk_blank_description_mapping.json", "w") as f:
        json.dump(descriptions, f, sort_keys=True, ensure_ascii=False, indent=4)
    
extract_descriptions(DF)       

In [ ]:
def create_categories_column(df: pd.DataFrame):
    df = df.sort_values("date").reset_index(drop=True)
    file = MAPPINGS_PATH / "completed" / "revolut_dkk.json"
    with open(file) as f:
        mappings = json.load(f)

    df["categories"] = df["description"].map(mappings)

    return df

DF = create_categories_column(DF)

In [ ]:
print(DF["date"].min(), end="\n\n\n")
print(DF["date"].max(), end="\n\n\n")
print(DF.dtypes, end="\n\n\n")
print(DF.isna().sum().to_dict(), end="\n\n\n")
display(DF.head(5))
display(DF.tail(3))

In [ ]:
def create_parquet(df: pd.DataFrame, parquet_name: str):
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / parquet_name)

create_parquet(DF, "revolut_dkk_2024-08_2026-05.parquet")